In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

- Data Source : https://www.kaggle.com/datasets/miguelcorraljr/ted-ultimate-dataset/data
- Exact dataset : ted_talks_en.csv

In [94]:
df = pd.read_csv('../data/raw/ted_talks_en.csv')

print("Rows:", df.shape[0], "Cols:", df.shape[1])
print(df.columns.tolist())
df.head(3)

Rows: 4005 Cols: 19
['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript']


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,native_lang,available_lang,comments,duration,topics,related_talks,url,description,transcript
0,1,Averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25,2006-06-27,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272.0,977,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre..."
1,92,The best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22,2006-06-27,TED2006,en,"['ar', 'az', 'bg', 'bn', 'bs', 'cs', 'da', 'de...",628.0,1190,"['Africa', 'Asia', 'Google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac..."
2,7,Simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24,2006-06-27,TED2006,en,"['ar', 'bg', 'de', 'el', 'en', 'es', 'fa', 'fr...",124.0,1286,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun..."


In [95]:
required_columns = ["talk_id", "title", "speaker_1", "occupations", "description",
        "topics", "transcript", "url", "views", "duration",
        "published_date", "event"]

df = df[[c for c in required_columns if c in df.columns]]
print("Shape after drop:", df.shape)
print(df.columns.tolist())

Shape after drop: (4005, 12)
['talk_id', 'title', 'speaker_1', 'occupations', 'description', 'topics', 'transcript', 'url', 'views', 'duration', 'published_date', 'event']


In [96]:
# rename columns
df = df.rename(columns={
    "speaker_1": "speaker",
    "topics": "tags",
    "occupations": "speaker_occupation",
})

In [97]:
import ast

# extracting the occupation of the speaker from list of occupations of all speakers and converting to comma-separated string
def get_first_speaker_occupation(val):
  try:
    parsed = ast.literal_eval(val)
    if isinstance(parsed, dict) and 0 in parsed:
      return parsed[0]
  except (ValueError, SyntaxError):
    pass
  return val

df["speaker_occupation"] = df["speaker_occupation"].apply(get_first_speaker_occupation)
# list to comma separated strings for occupations and topics
df["speaker_occupation"] = df["speaker_occupation"].apply(lambda x: ", ".join(x) if isinstance(x, list) else str(x))

In [98]:
import ast 

# converting the string of a list of tags to a comma-separated string
def comma_separated_string_tags(val):
  try:
    parsed = ast.literal_eval(val)
    if isinstance(parsed, list):
      return ", ".join(parsed)
  except (ValueError, SyntaxError):
    pass
  return str(val)

df["tags"] = df["tags"].apply(comma_separated_string_tags)
# df["tags"] = df["tags"].apply(lambda x: ", ".join(x) if isinstance(x, list) else str(x))

In [99]:
print("Duplicate rows:", df.duplicated().sum())

print("Shape:", df.shape)
print(df.columns.tolist())
df.head(3)

Duplicate rows: 0
Shape: (4005, 12)
['talk_id', 'title', 'speaker', 'speaker_occupation', 'description', 'tags', 'transcript', 'url', 'views', 'duration', 'published_date', 'event']


,talk_id,title,speaker,speaker_occupation,description,tags,transcript,url,views,duration,published_date,event
0,1,Averting the climate crisis,Al Gore,climate advocate,With the same humor and humanity he exuded in ...,"alternative energy, cars, climate change, cult...","Thank you so much, Chris. And it's truly a gre...",https://www.ted.com/talks/al_gore_averting_the...,3523392,977,2006-06-27,TED2006
1,92,The best stats you've ever seen,Hans Rosling,global health expert; data visionary,You've never seen data presented like this. Wi...,"Africa, Asia, Google, demo, economics, global ...","About 10 years ago, I took on the task to teac...",https://www.ted.com/talks/hans_rosling_the_bes...,14501685,1190,2006-06-27,TED2006
2,7,Simplicity sells,David Pogue,technology columnist,New York Times columnist David Pogue takes aim...,"computers, entertainment, interface design, me...","(Music: ""The Sound of Silence,"" Simon & Garfun...",https://www.ted.com/talks/david_pogue_simplici...,1920832,1286,2006-06-27,TED2006


In [100]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4005 entries, 0 to 4004
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   talk_id             4005 non-null   int64
 1   title               4005 non-null   str  
 2   speaker             4005 non-null   str  
 3   speaker_occupation  4005 non-null   str  
 4   description         4005 non-null   str  
 5   tags                4005 non-null   str  
 6   transcript          4005 non-null   str  
 7   url                 4005 non-null   str  
 8   views               4005 non-null   int64
 9   duration            4005 non-null   int64
 10  published_date      4005 non-null   str  
 11  event               4005 non-null   str  
dtypes: int64(3), str(9)
memory usage: 375.6 KB


In [101]:
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Unique talk IDs:", df["talk_id"].nunique())
print("Duplicate talk IDs:", df["talk_id"].duplicated().sum())

talk_id               0
title                 0
speaker               0
speaker_occupation    0
description           0
tags                  0
transcript            0
url                   0
views                 0
duration              0
published_date        0
event                 0
dtype: int64
Duplicate rows: 0
Unique talk IDs: 4005
Duplicate talk IDs: 0


In [102]:
df.to_csv("../data/raw/ted_talks_selected.csv", index=False)

print("Saved:", "../data/raw/ted_talks_selected.csv")

Saved: ../data/raw/ted_talks_selected.csv


## Dataset Inspection Report

**1. What is the name and source of the dataset?** <br>
The dataset is **TED Ultimate Dataset**, downloaded from Kaggle:
https://www.kaggle.com/datasets/miguelcorraljr/ted-ultimate-dataset/data
The exact file used is `ted_talks_en.csv` (English TED Talks).

**2. How many rows and columns does it contain?** <br>
The dataset contains **4,005 rows** and **12 columns** after selecting only the required fields.
Columns: `talk_id`, `title`, `speaker`, `speaker_occupation`, `description`, `tags`, `transcript`, `url`, `views`, `duration`, `published_date`, `event`.

**3. How many unique TED Talks are present?**
There are **4,005 unique TED Talks**, confirmed by:
- `df["talk_id"].nunique()` -> 4,005
- `df["talk_id"].duplicated().sum()` -> 0  <br>
Every row represents a distinct talk with a unique `talk_id`.

**4. Which columns contain text that could be used for recommendations?** <br>
Text-bearing fields suitable for content-based recommendation:
- `title` - short descriptive text
- `description` - short summary of the talk
- `tags` - comma-separated topic keywords
- `transcript` - full spoken text (most information-rich)
- `speaker_occupation` - short context about the speaker (weak signal)
- `speaker` - speaker name (not useful for content similarity, but useful for display/filtering)

`transcript`, `tags`, and `description` are the strongest signals and will be combined into a `combined_text` field in Task 3.

**5. How many values are missing in the important columns?** <br>
`df.isnull().sum()` returns **0 missing values** for every column (all 12 columns have 4,005 non-null entries). No `NaN` or empty numeric fields were detected.
Note: this counts only `NaN` values. Empty strings (e.g., blank `tags` stored as `""` or `"[]"`) will be re-checked explicitly during Task 2 EDA and Task 3 cleaning.

**6. Are there duplicate rows or repeated talk identifiers?** <br>
No. `df.duplicated().sum()` -> **0 duplicate rows**, and `df["talk_id"].duplicated().sum()` → **0 duplicate talk IDs**. The dataset is clean on both counts.

**7. Are transcripts available for all talks, some talks, or none?** <br>
Transcripts are available for **all 4,005 talks**. The `transcript` column has 4,005 non-null entries of type `str`. However, some transcripts may be very short (e.g., music-only or performance pieces) - this will be quantified in Task 2 by computing word counts.

**8. What limitations might affect the recommendation system?** <br>
- **English-only**: the dataset contains only English-language talks (`ted_talks_en.csv`), so recommendations will be limited to English content and may not generalize across languages.
- **Possible very short transcripts**: talks that are mostly music or performance may have very few words, reducing the effectiveness of text-based similarity.
- **Tags are free-form**: tags are not standardized categories, so tag-based filtering may be noisy.
- **Popularity bias**: `views` is highly skewed toward a few famous talks, which could bias any popularity-weighted re-ranking.
- **No user interaction data**: no user ratings, watch history, or clicks - so only content-based (not collaborative) recommendation is possible.
- **Text noise**: transcripts include filler words, laughter cues `(Laughter)`, applause cues, and speaker names - these need preprocessing (Task 3) to avoid diluting similarity.

**9. Is the dataset suitable for a first content-based recommendation prototype? Explain why.** <br>
Yes. The dataset is well suited for a first content-based prototype because:
- It has **full transcripts** for every talk — the richest possible text signal for TF-IDF / cosine similarity.
- It provides **tags and descriptions**, allowing a meaningful `combined_text` field that mixes topical keywords with natural-language summaries.
- It is **clean**: 0 missing values, 0 duplicate rows, 0 duplicate IDs.
- It has a **reasonable size** (4,005 talks) that is large enough to produce meaningful recommendations yet small enough to run on a laptop.
- It includes metadata (`views`, `duration`, `event`, `published_date`) for optional re-ranking and evaluation.
Overall, the dataset satisfies every requirement for the content-based prototype phase and no additional data collection is needed.